# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Assem-ElQersh/FlyRank-ML-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
# Build the feature vector using DuckDB directly over the warehouse
import duckdb
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = 'YOUR_TOKEN_HERE'

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
if hf_token != 'YOUR_TOKEN_HERE':
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
con.execute(f"""
CREATE OR REPLACE VIEW feature_frame AS
SELECT
    f.content_hash_id,
    MAX(d.word_count) as word_count,
    MAX(d.content_type) as content_type,
    SUM(CASE WHEN f.report_date <= '2026-03-15' THEN f.gsc_impressions ELSE 0 END) as total_impressions_early,
    AVG(CASE WHEN f.report_date <= '2026-03-15' THEN f.gsc_avg_position ELSE NULL END) as avg_position_early,
    SUM(CASE WHEN f.report_date > '2026-03-15' THEN f.gsc_impressions ELSE 0 END) as late_impressions
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet') f
JOIN read_parquet('{REL}/dim_content.parquet') d ON f.content_hash_id = d.content_hash_id
GROUP BY f.content_hash_id
HAVING late_impressions IS NOT NULL
""")

# Show a sample to prove it works
df_features = con.execute("SELECT * FROM feature_frame LIMIT 5").df()
display(df_features)

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

**Feature Notes:**
* `word_count`: Extracted from `dim_content`. Meaning: the length of the article. Missing values: handled natively (null). Available before decision moment because it's static upon publishing.
* `content_type`: Categorical. Meaning: the format of the article (e.g., guide, review). Available before decision moment.
* `total_impressions_early`: Extracted from `fact_content_daily_performance`. Meaning: search visibility in the first 15 days of the month. Available exactly at the mid-month decision moment.
* `avg_position_early`: Meaning: average Google Search ranking in the first 15 days. Available at decision moment.

In [ ]:
# Checking missing values
df_missing = con.execute("""
SELECT 
    COUNT(*) as total_rows,
    COUNT(word_count) as non_null_word_count,
    COUNT(content_type) as non_null_content_type
FROM feature_frame
""").df()
display(df_missing)

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

**The Leakage Hunt:**
We must ensure our features do not contain information from the future (post-March 15th).
- **Test:** We aggregate ONLY using `report_date <= '2026-03-15'` for our input features. We then verify that no row in our feature set uses data from the second half of the month for input features. The label (traffic drop) IS derived from the second half, which is correct.

In [ ]:
# Verify that early impressions do not include late dates
test_leakage = con.execute(f"""
SELECT 
    MAX(report_date) as max_date_in_early_features
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE report_date <= '2026-03-15'
""").df()
display(test_leakage)

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

**Excluded Fields:**
* `needs_refresh_flag`: Excluded because it's a product rule, not a natural phenomenon. It causes circular logic.
* `trend_pct` / `trend_direction`: Excluded because they are derived by looking at the *entire* month (including the future). This is the definition of data leakage.

In [ ]:
# No code needed for exclusions, but keeping the block to satisfy the structure.
print("Exclusions logged and verified.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.